# Binôme B — Pipeline d'inférence et déploiement académique

**Phase CRISP-DM : Deployment.** Ce notebook présente l'interface locale du modèle figé, l'entraînement reproductible et l'évaluation par scripts. Il ne crée ni API, ni conteneur, ni infrastructure de production, conformément au périmètre demandé.

> Le split test a été consulté lors d'une itération antérieure ; les résultats associés sont exploratoires. Ce notebook ne règle aucun paramètre avec ces résultats.

## 1. Contrat d'entrée et chargement du modèle

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise FileNotFoundError("Racine du dépôt introuvable.")
sys.path.insert(0,str(ROOT / "src"))
from roadanomaly import data_pipeline as dp
from roadanomaly.anomaly_detector import AnomalyDetector

CONFIG_PATH=ROOT / "models" / "B_best_config.json"
if not CONFIG_PATH.is_file():
    raise FileNotFoundError("Exécuter les notebooks Modeling et Evaluation pour créer la configuration et le modèle.")
detector=AnomalyDetector(CONFIG_PATH)
print("Résolution attendue (H,W):",detector.image_size)
print("sigma:",detector.sigma,"| seuil:",detector.threshold)
print("L'entrée de predict est une image RGB float32 dans [0,1], déjà redimensionnée comme dans le pipeline partagé.")

## 2. Prédiction et vérification des sorties

In [ ]:
val_frames,x_val,y_val=dp.load_split("val",detector.image_size)
example=detector.predict(x_val[0])
assert example["mask"].shape==detector.image_size and example["mask"].dtype==np.uint8
assert set(np.unique(example["mask"])).issubset({0,1})
assert example["score_map"].shape==detector.image_size and example["score_map"].dtype==np.float32
assert np.isfinite(example["score_map"]).all()
print("Frame:",val_frames[0],"| pixels anomalie prédits:",int(example["mask"].sum()))

## 3. Explication Grad-CAM

`explain` calcule le Grad-CAM du bottleneck pour la MSE brute dans la région prédite. Il ne reçoit pas de masque terrain et ne l'utilise pas ; la heatmap est un outil de diagnostic, **pas une explication causale**.

In [ ]:
heatmap=detector.explain(x_val[0])
assert heatmap.shape==detector.image_size and np.isfinite(heatmap).all()
assert heatmap.min()>=0 and heatmap.max()<=1+1e-6
fig,axes=plt.subplots(1,4,figsize=(15,4),tight_layout=True)
axes[0].imshow(x_val[0]); axes[0].set_title("Image")
axes[1].imshow(example["mask"],cmap="gray"); axes[1].set_title("Masque prédit")
axes[2].imshow(example["score_map"],cmap="magma"); axes[2].set_title("Carte de score")
axes[3].imshow(x_val[0]); axes[3].imshow(heatmap,cmap="jet",alpha=0.45); axes[3].set_title("Grad-CAM (non causal)")
for axis in axes: axis.axis("off")
plt.show()

## 4. Entraînement reproductible

Le script `scripts/train.py` repart de train/validation, extrait uniquement des patches normaux pour l'entraînement, fixe la seed et active les opérations déterministes. Il calibre σ et le seuil exclusivement sur validation puis exporte le modèle et la configuration JSON. Exécuter depuis la racine du dépôt ; `--loss` accepte `mse`, `mae` et `ssim`.

In [ ]:
train_command=[
    sys.executable,str(ROOT / "scripts" / "train.py"),
    "--data-dir",str(ROOT / "datasets" / "RoadAnomaly"),
    "--manifest",str(ROOT / "splits" / "split_manifest.csv"),
    "--model-output",str(ROOT / "models" / "B_autoencoder.keras"),
    "--config-output",str(ROOT / "models" / "B_best_config.json"),
    "--seed","1337","--epochs","150","--loss","mse",
]
print("Commande (non exécutée automatiquement) :")
print(" ".join(train_command))
print("Lancer la cellule uniquement si vous souhaitez réentraîner le modèle.")

## 5. Évaluation d'un split

`evaluate.py` calcule AuPRC, AUROC, AUPRO@0.3, métriques pixel moyennes et agrégées, objets, tailles et faux positifs, puis écrit un JSON. Le choix du split est explicite ; l'avertissement exploratoire est affiché si le split test est demandé.

In [ ]:
evaluation_command=[
    sys.executable,str(ROOT / "scripts" / "evaluate.py"),
    "--config",str(CONFIG_PATH),"--split","val",
    "--manifest",str(ROOT / "splits" / "split_manifest.csv"),
    "--data-dir",str(ROOT / "datasets" / "RoadAnomaly"),
    "--output",str(ROOT / "reports" / "B_validation_metrics.json"),
]
print(" ".join(evaluation_command))
print("Pour exécuter : subprocess.run(evaluation_command, cwd=ROOT, check=True).")

## 6. Non-régression et démonstration sur trois images de validation

In [ ]:
test_command=[sys.executable,"-m","pytest",str(ROOT / "tests" / "test_pipeline.py"),"-q"]
print("Test ciblé :"," ".join(test_command))
print("Les tests nécessitent les artefacts générés par le notebook Modeling.")

fig,axes=plt.subplots(3,3,figsize=(13,10),tight_layout=True)
for row in range(3):
    image=x_val[row]
    prediction=detector.predict(image)
    axes[row,0].imshow(image); axes[row,0].set_title(f"{val_frames[row]} — image")
    axes[row,1].imshow(prediction["mask"],cmap="gray"); axes[row,1].set_title("Masque binaire")
    axes[row,2].imshow(prediction["score_map"],cmap="magma"); axes[row,2].set_title(f"Score (σ={prediction['sigma']}, seuil={prediction['threshold']:.3f})")
for axis in axes.flat: axis.axis("off")
plt.show()

## 7. Limites

Le détecteur attend l'image déjà décodée, redimensionnée à la taille configurée et normalisée par le pipeline partagé. La sortie de `explain` est une attribution Grad-CAM de la MSE dans la région prédite, sans garantie causale. Les tests de non-régression utilisent les métriques par image enregistrées à l'entraînement et doivent être exécutés après génération des artefacts.